In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%pip -q install -U trl transformers datasets peft accelerate bitsandbytes

In [ ]:
%pip install flash-attn --no-build-isolation   # for A100 when we use flash-attn

In [ ]:
!git clone https://github.com/DimitrisKu/Active-Reading--Pattern-Recognition.git

import os

%cd /content/Active-Reading--Pattern-Recognition

os.getcwd()

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from datasets import load_dataset
from itertools import chain
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from pathlib import Path
from datasets import concatenate_datasets


RUN_DIR = "/content/drive/MyDrive/qa_finetune/qlora_runs/qwen4b_qa_checkpoints"
SAVE_DIR = "/content/drive/MyDrive/qa_finetune/qlora_runs/final_qlora_adapter"


# --- Config ---
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
ORIGINAL_DATA_PATH = "/content/Active-Reading--Pattern-Recognition/Finetune_Datasets/financebench/original_mixin.jsonl"
DATA_PATH = "/content/Active-Reading--Pattern-Recognition/Finetune_Datasets/financebench/active_reading_dataset.jsonl"
MAX_SEQ_LENGTH = 1024
LEARNING_RATE = 2e-4


# --- Dataset ---
act_read_dataset = load_dataset("json", data_files=DATA_PATH, split="train")
original_dataset = load_dataset("json", data_files=ORIGINAL_DATA_PATH, split="train")


dataset = concatenate_datasets([act_read_dataset, original_dataset]).shuffle(seed=42)


# --- Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token



def to_text(ex):
    if "active_reading" in ex and ex["active_reading"]:
        txt = ex["active_reading"].strip()
    else:
        txt = ex.get("text", "")
        txt = txt.strip()
    return {"text": txt + tokenizer.eos_token}


dataset = dataset.map(to_text, remove_columns=dataset.column_names)



# --- 4-bit Quantization Config ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.float16, # for T4-GPU
    bnb_4bit_compute_dtype=torch.bfloat16, # for A100
    bnb_4bit_use_double_quant=True,
)

# --- Load Model (QLoRA) ---
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="flash_attention_2", # for A100
)

# --- Prep for k-bit training ---
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.gradient_checkpointing_enable()

# --- LoRA ---
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()



def group_texts(examples):
    concatenated = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // MAX_SEQ_LENGTH) * MAX_SEQ_LENGTH

    result = {
        k: [t[i:i + MAX_SEQ_LENGTH] for i in range(0, total_length, MAX_SEQ_LENGTH)]
        for k, t in concatenated.items()
    }
    return result


def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        add_special_tokens=False,
    )
    return tokenized


tokenized = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
    num_proc=2
)


lm_dataset = tokenized.map(
    group_texts,
    batched=True,
    num_proc=2
)


# --- Training Args ---
training_args = TrainingArguments(
    output_dir=RUN_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=LEARNING_RATE,
    max_steps=330,
    # fp16=True, # for T4-GPU
    bf16=True, # for A100
    tf32=True, # for A100
    logging_steps=10,
    save_steps=40,
    save_total_limit=2,
    report_to="none",
)

# --- Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print(" Starting QLoRA repetition fine-tuning...")
#trainer.train(resume_from_checkpoint=True)
trainer.train()

print(" Saving adapter...")
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)